# ARCHIVE - Phase 0-3 settled experiments (Orbit-Wars-era)

These cells answered questions we have already settled; they are parked here so the
working notebook (`colab_selfplay.ipynb`) only spends L4 cycles on open questions.

Settled here (do not re-run unless re-opening the question):
1. **No-collapse demo** - the stabilized `small` self-play recipe climbs without the
   P2.5 blow-up.
2. **Distributed-collector throughput** - W-sweep; dec/s still rose to W=48 on the L4
   (8: 673 -> 48: 2545 dec/s), so run the probes at `--workers 48`.
3. **Recipe-v2 stability sanity** on `small` - 3 seeds, no collapse, vs-random 75-98%.
4. **The definitive option-rank A/B** - **SETTLED: KEEP `use_option_rank=True`**
   (commit `4ac5e5f`). Re-running it burns hours on a closed question.

Run cell 1 (mount + clone) first if you actually need to reproduce any of these.

In [ ]:
# Mount Drive (artifact persistence), then clone (or update) the repo and cd in.
import os

from google.colab import drive
drive.mount('/content/drive')

# Everything the runs produce is written under here so it survives the session and
# can be pulled back with scripts/download_artifacts.py. Keep this name in sync with
# DRIVE_BASE in that script ('ptcg_outputs').
DRIVE_OUTPUT = '/content/drive/MyDrive/ptcg_outputs'
for sub in ['', '/logs', '/checkpoints_sp_small', '/ablation_sp']:
    os.makedirs(DRIVE_OUTPUT + sub, exist_ok=True)

REPO_URL = "https://github.com/oshbocker/pokemon_tcg_ai_battle.git"
REPO_DIR = "/content/pokemon_tcg_ai_battle"
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only
%cd {REPO_DIR}

assert os.path.isfile("agent/cg/libcg.so"), (
    "agent/cg/libcg.so missing - push it to the repo (see .gitignore exception)."
)
import torch
print("repo:", os.getcwd(), "| Drive:", DRIVE_OUTPUT)
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())


In [ ]:
# Hardware topology - record this alongside the results.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "(no GPU runtime selected)"
!nproc
!lscpu | grep -E "^CPU\(s\):|Thread\(s\) per core|Core\(s\) per socket|Model name"
!free -h | head -2


## 1. No-collapse demo (acceptance b)

A `small` self-play run on the distributed collector with every Phase-3 stabilizer
on. Watch `kl` stay bounded (a trailing `*` = the KL early-stop fired that iter, by
design) and the `gate vs best` win-rate ratchet up via promotions, instead of the
late-training collapse the unstabilized P2.5 loop showed. ~40 iters is a quick
read; bump `--iters` for a real curve.

Checkpoints go straight to Drive (`--out`), so this **resumes** if the session
dies; the log is tee'd locally then copied to Drive at the end.

In [ ]:
# W=32 = throughput sweet spot from the cell-2 sweep (workers mostly wait on the GPU,
# so oversubscribing the 12 vCPUs still helps); games/iter 128 keeps the batch >=48.
LOG = '/content/colab_selfplay_nocollapse.txt'
!python scripts/train_selfplay.py --size small --collector dist --workers 32 \
    --opponent self --league --league-kaggle romanrozen_v10 \
    --iters 40 --games-per-iter 128 --epochs 2 --minibatch 256 \
    --lr 3e-4 --lr-final 5e-5 --target-entropy 0.1 --target-kl 1.5 \
    --gate --gate-every 5 --gate-games 200 --gate-threshold 0.55 \
    --eval-every 5 --eval-games 120 --eval-opponents random,first,heuristic \
    --device cuda --out {DRIVE_OUTPUT}/checkpoints_sp_small 2>&1 | tee {LOG}
!cp {LOG} {DRIVE_OUTPUT}/logs/   # persist the log to Drive


## 2. Distributed-collector throughput sanity

Times a few collect-only iterations and reports decisions/s for the batched GPU
loop. Sweeps W = 8 → 48 to find where dec/s plateaus on the L4 — that peak is
the W to set for the A/B. (With the vectorised `act()`, per-decision latency
dropped, so the sweet spot may now sit lower than the first run's still-climbing
curve; if it's still rising at 48, tell me and we widen further.)

In [ ]:
import sys, time
sys.path.insert(0, "src")
import torch
from ptcg_battle.model import SIZE_BANDS, PtcgNet
from ptcg_battle.ppo import load_deck
from ptcg_battle.dist_collector import DistributedCollector

deck = load_deck()
model = PtcgNet(SIZE_BANDS["small"]).cuda().eval()
lines = []
for W in (8, 12, 16, 24, 32, 48):
    col = DistributedCollector(deck, n_workers=W)  # self-play = collect()'s default (league=None)
    try:
        col.collect(model, n_games=W, device="cuda", seed=1)  # warm up workers
        t0 = time.time()
        buf = col.collect(model, n_games=8 * W, device="cuda", seed=2)
        dt = time.time() - t0
    finally:
        col.close()
    line = f"W={W:>3}  decisions={len(buf):>6}  {len(buf)/dt:8.0f} dec/s  ({dt:.1f}s)"
    print(line); lines.append(line)

with open(f"{DRIVE_OUTPUT}/logs/colab_dist_throughput.txt", "w") as f:
    f.write("\n".join(lines) + "\n")


## 2b. Recipe-v2 stability sanity on `small` (go/no-go before the A/B)

The tiny-model local validation passed (vs-random 90-93%, no collapse), but tiny
keeps entropy high on its own — it never exercised the controller's *raise the
bonus* path, which is the actual `small` failure mode (ON arms collapsed entropy
to ~0 and fell below random). This cell is that test: 3 short `small` self-play
seeds with recipe v2 (adaptive entropy + KL circuit breaker).

**PASS** (then run the A/B in section 3):
- `ent` stays >= ~0.03 every seed; if it dips, `entc` rises toward 0.2 (the
  controller doing its job).
- `eval: random` clears >75% on all 3 seeds, clustered — no seed cratering to ~40%.

**FAIL** -> raise `--target-entropy` (and/or the controller's ceiling) and re-check
before spending hours on the A/B.

In [ ]:
# 3 short `small` self-play seeds; watch ent / entc and eval-vs-random per seed.
for s in (0, 1, 2):
    print(f"\n######## seed {s} ########")
    !python scripts/train_selfplay.py --size small --collector dist --workers 32 \
        --opponent self --league --league-kaggle romanrozen_v10 \
        --iters 35 --games-per-iter 128 --epochs 2 --minibatch 256 \
        --lr 3e-4 --lr-final 5e-5 --target-entropy 0.1 --target-kl 1.5 \
        --gate --gate-every 5 --gate-games 120 \
        --eval-every 5 --eval-games 120 --eval-opponents random --seed {s} \
        --out {DRIVE_OUTPUT}/sanity_s{s} 2>&1 | tee {DRIVE_OUTPUT}/logs/colab_small_sanity_s{s}.txt


## 3. The definitive option-rank A/B (self-play)

Trains ON vs OFF arms via stabilized self-play, then judges the best checkpoints on
the honest suite + head-to-head at high n. **Multi-hour for `small`.** Start with
`--seeds 2`; raise it (and `--iters`) if the verdict comes back `INCONCLUSIVE`. The
printed RECOMMENDATION + table go into `rl_research/ABLATION_OPTION_RANK.md`.

Per-arm checkpoints + eval CSVs are written under `--out` on Drive; the log is
copied to Drive at the end.

In [ ]:
LOG = '/content/colab_ablation_selfplay.txt'
!python scripts/ablate_option_rank_selfplay.py \
    --size small --workers 32 --seeds 2 --iters 80 --games-per-iter 128 \
    --epochs 2 --minibatch 256 --lr 3e-4 --lr-decay 0.17 --target-entropy 0.1 --target-kl 1.5 \
    --league-kaggle romanrozen_v10 \
    --sel-every 5 --sel-n 200 \
    --eval-n 2000 --device cuda --out {DRIVE_OUTPUT}/ablation_sp --verbose \
    2>&1 | tee {LOG}
!cp {LOG} {DRIVE_OUTPUT}/logs/   # persist the log to Drive


## Results are on Drive — pull them to the laptop

Everything the runs produced lives under `MyDrive/ptcg_outputs/` (checkpoints, eval
CSVs, and the `logs/colab_*.txt` records). Nothing needs to be git-pushed from here.
The cell below just confirms what landed; back on the laptop, fetch it with rclone:

```bash
uv run python scripts/download_artifacts.py --logs                 # colab_*.txt -> rl_research/
uv run python scripts/download_artifacts.py --run ablation_sp      # A/B ckpts + CSVs -> outputs/
uv run python scripts/download_artifacts.py --run checkpoints_sp_small
```

Then paste the A/B table + RECOMMENDATION into `ABLATION_OPTION_RANK.md` and commit
the logs. (One-time rclone `gdrive` remote setup is in CLAUDE.md.)

In [ ]:
# Confirm what persisted to Drive.
!ls -lhR {DRIVE_OUTPUT} | sed -n '1,80p'
